# 决策树第三课：条件熵与信息增益

本课目标：把条件熵讲成一笔能手算清楚的账。你需要记住一句话：**条件熵 = 按某个特征分组以后，每个小组内部熵的加权平均。**

## 1. 先别急着看公式：条件熵到底在问什么？

假设我们要预测“今天是否打球”。如果完全不看任何特征，只看全部样本里的结果比例，这叫分裂前的信息熵 $H(D)$。

现在我们多知道一个条件，比如“天气”。天气可能是“晴”或者“雨”。于是原来的数据会被分成两个小组：晴天组、雨天组。

条件熵 $H(D|天气)$ 问的是：**已经知道天气以后，是否打球这个结果还剩多少不确定性？**

所以它不是凭空来的新概念，而是在问：按天气分组后，每个组里还乱不乱？整体平均起来还乱不乱？

信息熵是干什么的？它是衡量数据集中结果的混乱程度。条件熵是干什么的？它是衡量在已知某个特征的情况下，数据集的混乱程度。

## 2. 原始数据：10 条打球记录

| 编号 | 天气 | 湿度 | 是否打球 |
| ---: | --- | --- | --- |
| 1 | 晴 | 高 | 否 |
| 2 | 晴 | 高 | 否 |
| 3 | 晴 | 正常 | 是 |
| 4 | 晴 | 正常 | 是 |
| 5 | 晴 | 高 | 是 |
| 6 | 晴 | 正常 | 是 |
| 7 | 雨 | 高 | 否 |
| 8 | 雨 | 正常 | 否 |
| 9 | 雨 | 高 | 否 |
| 10 | 雨 | 正常 | 是 |

全部 10 条里：

- 打球“是”：5 条
- 打球“否”：5 条

所以分裂前很混乱，一半是一半是否：

$$H(D)=-(\frac{5}{10}\log_2\frac{5}{10}+\frac{5}{10}\log_2\frac{5}{10})=1$$

## 3. 用“天气”分组以后，先数每个小组

我们按照“天气”这个特征把数据切开。

| 天气分组 | 样本数 | 是 | 否 | 这个组里的直觉 |
| --- | ---: | ---: | ---: | --- |
| 晴 | 6 | 4 | 2 | 更偏向“是”，但还不是纯的 |
| 雨 | 4 | 1 | 3 | 更偏向“否”，但也不是纯的 |

这一步非常关键：**条件熵不是只看分了几个组，而是看每个组内部还剩多少混乱。**

如果某个组全是“是”或者全是“否”，这个组的熵就是 0，因为它完全确定。

如果某个组一半“是”、一半“否”，这个组的熵接近 1，因为它最混乱。

## 4. 手算晴天组的熵

晴天组一共有 6 条，其中 4 条“是”、2 条“否”。

所以在晴天组里：

- $P(是)=\frac{4}{6}$
- $P(否)=\frac{2}{6}$

代入信息熵公式：

$$H(晴天组)=-(\frac{4}{6}\log_2\frac{4}{6}+\frac{2}{6}\log_2\frac{2}{6})$$

计算结果约为：

$$H(晴天组) \approx 0.9183$$

这个数为什么比较高？因为 4:2 虽然偏向“是”，但还没有到非常确定。你看到天气是晴，只能说“更可能打球”，不能拍板。

## 5. 手算雨天组的熵

雨天组一共有 4 条，其中 1 条“是”、3 条“否”。

所以在雨天组里：

- $P(是)=\frac{1}{4}$
- $P(否)=\frac{3}{4}$

代入信息熵公式：

$$H(雨天组)=-(\frac{1}{4}\log_2\frac{1}{4}+\frac{3}{4}\log_2\frac{3}{4})$$

计算结果约为：

$$H(雨天组) \approx 0.8113$$

它比晴天组的熵低一点，因为 1:3 比 4:2 更偏，结果更容易猜。雨天时更可能“不打球”。

## 6. 为什么条件熵要加权平均？

现在我们有两个组的熵：

- 晴天组熵：0.9183，占全部样本的 $\frac{6}{10}$
- 雨天组熵：0.8113，占全部样本的 $\frac{4}{10}$

条件熵就是这两个组的加权平均：

$$H(D|天气)=\frac{6}{10}\times0.9183+\frac{4}{10}\times0.8113$$

$$H(D|天气)=0.5510+0.3245=0.8755$$

为什么不是简单平均 $(0.9183+0.8113)/2$？因为晴天组有 6 条，雨天组只有 4 条。样本更多的组，对整体不确定性的影响应该更大。

所以条件熵这件事可以翻译成一句大白话：**我按天气问了一次问题之后，剩下的不确定性平均还有 0.8755。**

## 7. 信息增益：这一刀减少了多少混乱？

分裂前的信息熵是：

$$H(D)=1$$

按天气分裂后的条件熵是：

$$H(D|天气)=0.8755$$

所以天气带来的信息增益是：

$$Gain(D, 天气)=H(D)-H(D|天气)=1-0.8755=0.1245$$

解释成普通话就是：**问“天气是什么”这个问题，只让我们少了 0.1245 的混乱。**

如果另一个特征让混乱减少更多，决策树就更愿意先问另一个特征。

In [ ]:
from collections import Counter, defaultdict
from math import log2

def entropy(labels):
    """计算一组标签的信息熵。"""
    total = len(labels)
    counts = Counter(labels)
    result = -sum((count / total) * log2(count / total) for count in counts.values())
    return 0.0 if abs(result) < 1e-12 else result

def split_by_feature(data, feature):
    groups = defaultdict(list)
    for row in data:
        groups[row[feature]].append(row)
    return groups

def conditional_entropy(data, feature, target='是否打球'):
    total = len(data)
    groups = split_by_feature(data, feature)
    return sum(
        (len(rows) / total) * entropy([row[target] for row in rows])
        for rows in groups.values()
    )

def information_gain(data, feature, target='是否打球'):
    labels = [row[target] for row in data]
    return entropy(labels) - conditional_entropy(data, feature, target)

In [ ]:
data = [
    {'天气': '晴', '湿度': '高', '是否打球': '否'},
    {'天气': '晴', '湿度': '高', '是否打球': '否'},
    {'天气': '晴', '湿度': '正常', '是否打球': '是'},
    {'天气': '晴', '湿度': '正常', '是否打球': '是'},
    {'天气': '晴', '湿度': '高', '是否打球': '是'},
    {'天气': '晴', '湿度': '正常', '是否打球': '是'},
    {'天气': '雨', '湿度': '高', '是否打球': '否'},
    {'天气': '雨', '湿度': '正常', '是否打球': '否'},
    {'天气': '雨', '湿度': '高', '是否打球': '否'},
    {'天气': '雨', '湿度': '正常', '是否打球': '是'},
]

labels = [row['是否打球'] for row in data]
print(f"分裂前 H(D) = {entropy(labels):.4f}")

groups = split_by_feature(data, '天气')
total = len(data)
print("\n按天气分组后的每一笔账：")

for weather, rows in groups.items():
    group_labels = [row['是否打球'] for row in rows]
    group_entropy = entropy(group_labels)
    weight = len(rows) / total
    contribution = weight * group_entropy
    print(f"{weather}天组：{len(rows)}条，标签统计 {dict(Counter(group_labels))}")
    print(f"  组内熵 = {group_entropy:.4f}")
    print(f"  权重 = {len(rows)}/{total} = {weight:.1f}")
    print(f"  对条件熵的贡献 = {weight:.1f} × {group_entropy:.4f} = {contribution:.4f}")

weather_cond_entropy = conditional_entropy(data, '天气')
weather_gain = information_gain(data, '天气')
print(f"\nH(D|天气) = {weather_cond_entropy:.4f}")
print(f"Gain(D, 天气) = {weather_gain:.4f}")

## 8. 对比一下“湿度”这个特征

同样的算法也可以算“湿度”。决策树会比较不同特征的信息增益，优先选择信息增益更大的那个。

In [ ]:
for feature in ['天气', '湿度']:
    cond = conditional_entropy(data, feature)
    gain = information_gain(data, feature)
    print(f"特征：{feature}")
    print(f"  条件熵 H(D|{feature}) = {cond:.4f}")
    print(f"  信息增益 Gain(D, {feature}) = {gain:.4f}\n")

## 9. 本课小结

- 信息熵 $H(D)$：还没分组前，整体标签有多混乱。
- 条件熵 $H(D|A)$：按特征 $A$ 分组后，各组内部混乱程度的加权平均。
- 信息增益 $Gain(D,A)$：这次分组让混乱减少了多少。
- 决策树在当前节点会偏向选择信息增益更大的特征。

一句话抓住条件熵：**先分组，算每组熵，再按每组样本占比加权求和。**